In [9]:
##자막따기 좀 더 정확 버전

import os
import whisper

VIDEO_DIR = r"C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과"
OUTPUT_DIR = os.path.join(VIDEO_DIR, "output_txt")
os.makedirs(OUTPUT_DIR, exist_ok=True)

video_exts = [".mp4", ".avi", ".mov", ".mkv", ".m4v"]

print("Whisper 모델 로딩 중...")
model = whisper.load_model("small")   # base보다 추천

for file in os.listdir(VIDEO_DIR):
    if any(file.lower().endswith(ext) for ext in video_exts):
        video_path = os.path.join(VIDEO_DIR, file)
        filename = os.path.splitext(file)[0]

        print(f"\n처리 중: {file}")

        result = model.transcribe(
            video_path,
            language="ko",
            word_timestamps=True,
            verbose=False
        )

        output_path = os.path.join(OUTPUT_DIR, filename + ".txt")

        with open(output_path, "w", encoding="utf-8") as f:
            for seg in result["segments"]:
                start = seg["start"]
                end = seg["end"]
                text = seg["text"].strip()

                f.write(f"[{start:.2f}초 ~ {end:.2f}초] {text}\n")

                # 단어별 시간도 같이 저장
                if "words" in seg:
                    for word in seg["words"]:
                        wstart = word.get("start", None)
                        wend = word.get("end", None)
                        wtext = word.get("word", "").strip()

                        if wstart is not None and wend is not None:
                            f.write(f"    ({wstart:.2f} ~ {wend:.2f}) {wtext}\n")

                f.write("\n")

        print(f"저장 완료: {output_path}")

print("\n모든 영상 자막 추출 완료!")

Whisper 모델 로딩 중...

처리 중: “솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디_cut.mp4


100%|██████████| 27202/27202 [01:54<00:00, 237.00frames/s]


저장 완료: C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\output_txt\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디_cut.txt

처리 중: 공기업 공공기관 면접은 뭐가 다를까_cut.mp4


100%|██████████| 28114/28114 [01:20<00:00, 351.16frames/s]


저장 완료: C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\output_txt\공기업 공공기관 면접은 뭐가 다를까_cut.txt

처리 중: 남자 33세 4년 공백기 퇴직사유 경력 지원자를 보는 면접관의 속마음_cut.mp4


100%|██████████| 31985/31985 [01:37<00:00, 329.16frames/s]


저장 완료: C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\output_txt\남자 33세 4년 공백기 퇴직사유 경력 지원자를 보는 면접관의 속마음_cut.txt

처리 중: 면접관이 10초 만에 합격 불합격을 결정한 지원자의 특징_cut.mp4


100%|██████████| 24908/24908 [01:04<00:00, 386.74frames/s]

저장 완료: C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\output_txt\면접관이 10초 만에 합격 불합격을 결정한 지원자의 특징_cut.txt

모든 영상 자막 추출 완료!


In [ ]:
## 영상별로 답변 시간대 편집

import os
import subprocess
from pathlib import Path

# =========================
# 원본 영상 경로
# =========================
INPUT_VIDEO = r'C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\원본\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디.mp4'

# =========================
# 출력 폴더
# =========================
OUTPUT_DIR = r'C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 자를 구간들 (초 단위)
# =========================
segments = [
    (64.80, 89.76),
    (94.68, 110.46),
    (114.02, 127.64),
    (131.28, 145.06),
    (153.20, 173.94),
    (176.26, 180.20),
    (189.40, 205.32),
    (207.84, 225.42),
    (232.92, 252.44),
    (267.28, 284.20),
    (293.50, 314.98),
    (328.72, 365.76),
    (387.98, 410.90),
    (413.84, 432.62),
    (440.30, 448.80),
]

# =========================
# 파일명 설정
# =========================
input_path = Path(INPUT_VIDEO)
base_name = input_path.stem

temp_dir = os.path.join(OUTPUT_DIR, "temp_segments")
os.makedirs(temp_dir, exist_ok=True)

concat_list_path = os.path.join(temp_dir, "concat_list.txt")
final_output = os.path.join(OUTPUT_DIR, f"{base_name}_cut.mp4")

# =========================
# ffmpeg 실행 함수
# =========================
def run_ffmpeg(cmd):
    print("실행:", " ".join(cmd))
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffmpeg 실행 중 오류 발생")
    return result

# =========================
# 1. 각 구간 잘라내기
# =========================
segment_files = []

for i, (start, end) in enumerate(segments, start=1):
    duration = end - start
    seg_path = os.path.join(temp_dir, f"segment_{i:02d}.mp4")
    segment_files.append(seg_path)

    cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(start),
        "-i", INPUT_VIDEO,
        "-t", str(duration),
        "-c:v", "libx264",
        "-c:a", "aac",
        "-preset", "fast",
        "-crf", "23",
        seg_path
    ]
    run_ffmpeg(cmd)

# =========================
# 2. concat용 txt 만들기
# =========================
with open(concat_list_path, "w", encoding="utf-8") as f:
    for seg_file in segment_files:
        f.write(f"file '{seg_file.replace(os.sep, '/')}'\n")

# =========================
# 3. 잘라낸 구간 이어붙이기
# =========================
cmd_concat = [
    "ffmpeg",
    "-y",
    "-f", "concat",
    "-safe", "0",
    "-i", concat_list_path,
    "-c", "copy",
    final_output
]
run_ffmpeg(cmd_concat)

print("\n완료!")
print("최종 영상:", final_output)

실행: ffmpeg -y -ss 64.8 -i C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\원본\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디.mp4 -t 24.960000000000008 -c:v libx264 -c:a aac -preset fast -crf 23 C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\temp_segments\segment_01.mp4


Exception in thread Thread-33 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'cp949' codec can't decode byte 0xeb in position 2375: illegal multibyte sequence


실행: ffmpeg -y -ss 94.68 -i C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\원본\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디.mp4 -t 15.779999999999987 -c:v libx264 -c:a aac -preset fast -crf 23 C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\temp_segments\segment_02.mp4


Exception in thread Thread-35 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'cp949' codec can't decode byte 0xeb in position 2375: illegal multibyte sequence


실행: ffmpeg -y -ss 114.02 -i C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\원본\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디.mp4 -t 13.620000000000005 -c:v libx264 -c:a aac -preset fast -crf 23 C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\temp_segments\segment_03.mp4


...

실행: ffmpeg -y -ss 440.3 -i C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\원본\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디.mp4 -t 8.5 -c:v libx264 -c:a aac -preset fast -crf 23 C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\temp_segments\segment_15.mp4


Exception in thread Thread-61 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'cp949' codec can't decode byte 0xeb in position 2375: illegal multibyte sequence


실행: ffmpeg -y -f concat -safe 0 -i C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\temp_segments\concat_list.txt -c copy C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디_cut.mp4


Exception in thread Thread-63 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Dell3571\anaconda3\envs\whisper\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'cp949' codec can't decode byte 0xeb in position 2453: illegal multibyte sequence



완료!
최종 영상: C:\Users\Dell3571\Desktop\딥러닝\이형 유튜브 영상\클립결과\“솔직히 크게 관심 없었다” 면접관을 돌아서게 만든 한마디_cut.mp4
